# Geometry-V1 Q/K operational preflight handoff

This is a user-only operational preflight. It makes no watermark, transform, reliability, or scientific claim. **PREPARED_NOT_EXECUTABLE_FROM_COLAB**: the required execution exact has not been authorized for push, so do not run this notebook until the user separately authorizes a remotely retrievable exact.


In [ ]:
import os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V1'
EXECUTION_EXACT = 'beb5be85e53dda539e055ede98cda9da0ffd00c3'
RUNNER_MODULE = 'experiments.run_geometry_v1_qk_operational_preflight'
repo = pathlib.Path('/content/cegwm-geometry-v1-source')
if repo.exists():
    raise FileExistsError('fresh checkout path already exists')
subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
subprocess.run(['git', 'checkout', '--detach', EXECUTION_EXACT], cwd=repo, check=True)
head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
clean = subprocess.run(['git', 'status', '--porcelain'], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()
if head != EXECUTION_EXACT or clean:
    raise RuntimeError('execution checkout identity differs')
raise RuntimeError('PREPARED_NOT_EXECUTABLE_FROM_COLAB: push authorization is required before model or runner execution')


In [ ]:
# Run this cell only after explicit push and user execution authorization.
from google.colab import files, userdata
uploaded = files.upload()
if len(uploaded) not in (1, 2):
    raise ValueError('upload exactly one or two ordinary RGB images')
input_dir = pathlib.Path('/content/geometry-v1-inputs')
input_dir.mkdir(exist_ok=False)
paths = []
for name, data in uploaded.items():
    path = input_dir / pathlib.Path(name).name
    path.write_bytes(data)
    paths.append(str(path))
root_key = userdata.get('CEG_WM_ROOT_KEY')
hf_token = userdata.get('HF_TOKEN')
if not isinstance(root_key, str) or not root_key or not isinstance(hf_token, str) or not hf_token:
    raise RuntimeError('required Colab secret unavailable')
child_env = {name: value for name, value in os.environ.items() if all(marker not in name.upper() for marker in ('TOKEN', 'KEY', 'SECRET', 'PASSWORD', 'CREDENTIAL'))}
child_env['CEG_WM_ROOT_KEY'] = root_key
child_env['HF_TOKEN'] = hf_token
root_key = hf_token = ''
try:
    process = subprocess.run([sys.executable, '-m', RUNNER_MODULE, *paths], cwd=repo, env=child_env, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=False, timeout=1800)
    output = process.stdout[:4096].decode('utf-8', 'replace').strip()
    if process.returncode != 0 or len(process.stdout) > 4096:
        raise RuntimeError('sanitized runner failure')
    print(output)
finally:
    child_env.pop('CEG_WM_ROOT_KEY', None)
    child_env.pop('HF_TOKEN', None)
    child_env = None
    for path in input_dir.glob('*'):
        path.unlink()
    input_dir.rmdir()
